[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nishidaab26002-arch/sake-review-analysis/blob/main/programs/03_parse_review_text.ipynb)

# 03. 口コミ文字列の構造化

スクレイピングで取得した口コミは、ユーザー名・評価・本文・スペック情報・投稿日が**1本の文字列**に連結されています。
正規表現で各項目に分解し、分析に使える表形式にします。

抽出方式を3回改良しました。

| 版 | 方式 | 課題 |
|---|---|---|
| v1 | 先頭からユーザー名・評価を取り、キーワード位置で本文とスペックを分割 | 項目が混ざる |
| v2 | 末尾の日付を先に切り出し、原料米も抽出 | テイスト項目の欠損が大きい（下の品質確認参照） |
| v3（最終版） | **文字列の末尾から順に**確定キーワードを切り離す | 欠損率レポートで品質を確認 |

In [ ]:
# --- 実行環境の設定 ---------------------------------------------------------
# Colab: Google Driveをマウントし、BASE_DIR 配下を読み書きします。
# ローカル: 環境変数 SAKE_PROJECT_DIR でデータ置き場を指定できます。
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析'  # 自分のDriveのパスに変更
except ImportError:
    BASE_DIR = os.environ.get('SAKE_PROJECT_DIR', './sake_review_analysis')

CSV_DIR = f'{BASE_DIR}/口コミ_CSV'          # 収集・加工したCSV
for d in (CSV_DIR,):
    os.makedirs(d, exist_ok=True)

## v1: 先頭から抽出

In [ ]:
import pandas as pd
import re
import os

# 入力ファイルと出力ファイルのパス
INPUT_PATH = f'{CSV_DIR}/口コミ10000件.csv'
OUTPUT_PATH = f'{CSV_DIR}/saketime_analyzed_data.csv'

def split_review_text(text):
    """
    1本の文字列から各項目を正規表現で分離する関数
    """
    # ユーザー名と評価は構造的に先頭にあるため抽出
    # 例: "aki5.0田酒..." のような場合
    user_match = re.search(r'^(.*?)([0-9]\.[0-9])', text)
    user_name = user_match.group(1) if user_match else "不明"
    rating = user_match.group(2) if user_match else "不明"

    # 本文の抽出 (先頭からスペック情報が出てくるまでの間)
    # 特定名称、テイスト、ボディ等のキーワードが出現する位置を探す
    split_point = re.search(r'(特定名称|テイスト|ボディ|精米歩合|アルコール度数|\d{4}年\d{1,2}月\d{1,2}日)', text)
    body = text[len(user_name)+len(rating):split_point.start()] if split_point else text[len(user_name)+len(rating):]

    # 各スペック情報の抽出
    specific_name = re.search(r'特定名称(.*?)(?=テイスト|ボディ|精米歩合|$)', text)
    taste = re.search(r'テイスト(.*?)(?=ボディ|精米歩合|$)', text)
    body_type = re.search(r'ボディ(.*?)(?=精米歩合|アルコール|$)', text)
    post_date = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)', text)

    return {
        "user_name": user_name.strip(),
        "rating": rating,
        "review_body": body.strip(),
        "specific_name": specific_name.group(1).strip() if specific_name else "不明",
        "taste": taste.group(1).strip() if taste else "不明",
        "body_type": body_type.group(1).strip() if body_type else "不明",
        "post_date": post_date.group(1) if post_date else "不明"
    }

# CSVの読み込み
df = pd.read_csv(INPUT_PATH)

# 分解処理の実行
print("⚙️ レビューデータの分解を開始します...")
extracted_data = df['review'].apply(split_review_text)
df_extracted = pd.DataFrame(extracted_data.tolist())

# 元のデータと結合
df_final = pd.concat([df.drop(columns=['review']), df_extracted], axis=1)

# 保存
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"✅ 分解完了！新しいファイルを作成しました: {OUTPUT_PATH}")
print(df_final.head())

## v2: 日付を末尾から切り出し

In [ ]:
import pandas as pd
import re

# パスの設定
INPUT_PATH = f'{CSV_DIR}/saketime_robust_result.csv'
OUTPUT_PATH = f'{CSV_DIR}/saketime_analyzed_data_v2.csv'

def improved_split_review(text):
    # 1. 日付を最後から特定して切り出し
    date_match = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)$', text)
    post_date = date_match.group(1) if date_match else "不明"
    main_text = text[:date_match.start()] if date_match else text

    # 2. ユーザー名と評価の抽出（先頭固定）
    user_match = re.search(r'^(.*?)([0-9]\.[0-9])', main_text)
    user_name = user_match.group(1) if user_match else "不明"
    rating = user_match.group(2) if user_match else "不明"

    # 3. 本文とスペック情報の分離（キーワードを境界にする）
    # 特定名称、原料米、テイスト、ボディなどを探す
    spec_keywords = r'(特定名称|原料米|テイスト|ボディ)'
    split_pos = re.search(spec_keywords, main_text)

    body = main_text[len(user_name)+len(rating):split_pos.start()] if split_pos else main_text[len(user_name)+len(rating):]
    specs_part = main_text[split_pos.start():] if split_pos else ""

    # 4. 各スペックの抽出
    # 特定名称
    spec_name = re.search(r'特定名称(.*?)(?=原料米|テイスト|ボディ|$)', specs_part)
    specific_name = spec_name.group(1).strip() if spec_name else "不明"

    # 原料米 (rice_cultivar)
    rice = re.search(r'原料米(.*?)(?=特定名称|テイスト|ボディ|$)', specs_part)
    rice_cultivar = rice.group(1).strip() if rice else "不明"

    # テイスト
    taste = re.search(r'テイスト(.*?)(?=ボディ|$)', specs_part)
    taste_val = taste.group(1).strip() if taste else "不明"

    # ボディ
    body_val = re.search(r'ボディ(.*?)$', specs_part)
    body_type = body_val.group(1).strip() if body_val else "不明"

    return {
        "user_name": user_name.strip(),
        "rating": rating,
        "review_body": body.strip(),
        "specific_name": specific_name,
        "rice_cultivar": rice_cultivar,
        "taste": taste_val,
        "body_type": body_type,
        "post_date": post_date
    }

# CSVの処理
df = pd.read_csv(INPUT_PATH)
extracted_data = df['review'].apply(improved_split_review)
df_extracted = pd.DataFrame(extracted_data.tolist())

# 元データと結合して保存
df_final = pd.concat([df.drop(columns=['review']), df_extracted], axis=1)
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

print(f"✅ 分解完了！新しいファイルを作成しました: {OUTPUT_PATH}")

✅ 分解完了！新しいファイルを作成しました: /content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析/口コミ_CSV/saketime_analyzed_data_v2.csv


### v2の品質確認（欠損率）

In [ ]:
import pandas as pd

# CSVの読み込み
df = pd.read_csv(f'{CSV_DIR}/saketime_analyzed_data_v2.csv')

# 全件数
total_rows = len(df)

# 各カラムの「不明」または空値の数をカウント
# ※データによっては「不明」という文字列や、NaN(空)の可能性があるため両方チェックします
def count_missing(column):
    # '不明'という文字列、またはNaNをカウント
    missing_count = df[column].astype(str).str.contains('不明').sum() + df[column].isna().sum()
    return missing_count

print(f"📊 全口コミデータ数: {total_rows} 件\n")
print("--- [各項目の欠損(不明)数確認] ---")
target_columns = ['specific_name','rice_cultivar', 'taste', 'body_type', 'post_date']
for col in target_columns:
    count = count_missing(col)
    ratio = (count / total_rows) * 100
    print(f"{col}: {count} 件 ({ratio:.1f}%)")

# ついでに、specific_name に「原料米」が含まれてしまっている割合を確認
rice_in_spec = df['specific_name'].astype(str).str.contains('原料米').sum()
print(f"\n💡 修正が必要な箇所:")
print(f"specific_name に「原料米」が含まれている件数: {rice_in_spec} 件")

📊 全口コミデータ数: 167 件

--- [各項目の欠損(不明)数確認] ---
specific_name: 75 件 (44.9%)
rice_cultivar: 113 件 (67.7%)
taste: 161 件 (96.4%)
body_type: 56 件 (33.5%)
post_date: 0 件 (0.0%)

💡 修正が必要な箇所:
specific_name に「原料米」が含まれている件数: 0 件


## v3（最終版）: 末尾から順に切り離すパイプライン

日付 → 甘辛度 → ボディ感 → 酒の種類 → 原料米 → 特定名称 の順に末尾から切り離し、最後に残った先頭部分からユーザー名・評価・本文を分離します。
「原料米」という単語が本文中に出てくる誤検出は、文字数ガード（10文字以内）で防いでいます。

In [ ]:
import pandas as pd
import re

# 入出力パスの設定（元のプレーンデータから再スタートします）
INPUT_PATH = f'{CSV_DIR}/saketime_分類テスト用_result.csv'
OUTPUT_PATH = f'{CSV_DIR}/saketime_分類テスト用_analyzed.csv'

def extract_specs_backward(text):
    """
    文字列の後ろから順番に確定キーワードを探し、切り離していく抽出パイプライン
    """
    # 初期値
    extracted = {
        "user_name": "不明",
        "rating": "不明",
        "review_body": "不明",
        "specific_name": "不明",
        "sake_type": "不明", # 生酒などの「酒の種類」も追加抽出
        "rice_cultivar": "不明",
        "body_type": "不明",
        "amakara": "不明",
        "post_date": "不明"
    }

    current_text = str(text)

    # --------------------------------------------------
    # Step 1: 【日付】を末尾から切り離す
    # --------------------------------------------------
    date_match = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)$', current_text)
    if date_match:
        extracted["post_date"] = date_match.group(1)
        current_text = current_text[:date_match.start()].strip()

    # --------------------------------------------------
    # Step 2: 【甘辛度】を末尾から切り離す
    # --------------------------------------------------
    amakara_match = re.search(r'甘辛:(.*?)$', current_text)
    if amakara_match:
        extracted["amakara"] = amakara_match.group(1).strip()
        current_text = current_text[:amakara_match.start()].strip()

    # --------------------------------------------------
    # Step 3: 【ボディ感】を末尾から切り離す
    # --------------------------------------------------
    # 「テイストボディ:」または「ボディ:」に対応
    body_match = re.search(r'(?:テイスト)?ボディ:(.*?)$', current_text)
    if body_match:
        extracted["body_type"] = body_match.group(1).strip()
        current_text = current_text[:body_match.start()].strip()
        # 万が一「テイスト」という文字だけが残ってしまったら除去
        current_text = re.sub(r'テイスト$', '', current_text).strip()

    # --------------------------------------------------
    # Step 3.5: 【酒の種類】（生酒など）があれば切り離す（特定名称への混入防止）
    # --------------------------------------------------
    sake_type_match = re.search(r'酒の種類(.*?)$', current_text)
    if sake_type_match:
        extracted["sake_type"] = sake_type_match.group(1).strip()
        current_text = current_text[:sake_type_match.start()].strip()


    # --------------------------------------------------
    # Step 4: 【原料米】を末尾から切り離す（文字数ガード付き）
    # --------------------------------------------------
    rice_match = re.search(r'原料米(.*?)$', current_text)
    if rice_match:
        # 一旦、マッチした部分の文字列を rice_cultivar に入れる
        extracted["rice_cultivar"] = rice_match.group(1).strip()

        # 🌾 その文字数が10文字以内（本物のスペックの可能性が高い）場合のみ、文頭を削る
        if len(extracted["rice_cultivar"]) <= 10:
            current_text = current_text[:rice_match.start()].strip()
        else:
            # 10文字を超える場合は、本文中の「原料米」という単語に誤反応したとみなす
            # 取得結果を「不明」に戻し、文頭（本文）も削らずにスルーする
            extracted["rice_cultivar"] = "不明"



    # --------------------------------------------------
    # Step 5: 【特定名称】を末尾から切り離す
    # --------------------------------------------------
    spec_match = re.search(r'特定名称(.*?)$', current_text)
    if spec_match:
        extracted["specific_name"] = spec_match.group(1).strip()
        current_text = current_text[:spec_match.start()].strip()

    # --------------------------------------------------
    # Step 6: 残った先頭部分から【ユーザー名】【評価点】【本文】を分離
    # --------------------------------------------------
    # ユーザー名と点数 (例: "ユーザー名4.5" を分離)
    user_match = re.search(r'^(.*?)([0-9]\.[0-9]|-)', current_text)
    if user_match:
        extracted["user_name"] = user_match.group(1).strip()
        # 評価点が数字（4.5など）であれば取得、ハイフンなどの未評価記号なら不明へ
        rating_str = user_match.group(2).strip()
        extracted["rating"] = rating_str if rating_str != '-' else "不明"

        # 残ったすべてが純粋な口コミ本文
        extracted["review_body"] = current_text[user_match.end():].strip()
    else:
        # 万が一ユーザー名や点数が見つからなかった場合はそのまま本文に
        extracted["review_body"] = current_text

    return extracted

# =========================================================
# 実行処理
# =========================================================
print("⚙️ 最適化された抽出パイプラインを起動します...")

# 1. プレーンな取得結果データを読み込む
df = pd.read_csv(INPUT_PATH)
total_rows = len(df)

# 2. 関数を適用して分解
extracted_data = df['review'].apply(extract_specs_backward)
df_extracted = pd.DataFrame(extracted_data.tolist())

# 3. 元のデータ（prefecture, brand, maker, review_num, pricerange）と結合
df_final = pd.concat([df.drop(columns=['review']), df_extracted], axis=1)

# 4. 保存
df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

# =========================================================
# 最終的なデータ品質（欠損率）のレポート
# =========================================================
print(f"\n✅ 抽出完了！新しいファイルを作成しました: {OUTPUT_PATH}")
print(f"\n📊 --- [構造化データ 品質レポート (全{total_rows}件)] ---")
report_cols = ['specific_name', 'rice_cultivar', 'body_type', 'amakara', 'sake_type']

for col in report_cols:
    valid_count = (df_final[col] != '不明').sum()
    missing_count = total_rows - valid_count
    missing_ratio = (missing_count / total_rows) * 100
    print(f" 🔹 {col:<15} : 取得済 {valid_count:3}件 / 不明 {missing_count:3}件 (欠損率 {missing_ratio:.1f}%)")

print("\n💡 分析（Doc2Vec等）への活用準備が完全に整いました。")

⚙️ 最適化された抽出パイプラインを起動します...

✅ 抽出完了！新しいファイルを作成しました: /content/drive/MyDrive/作成したプログラム等/日本酒口コミ分析/口コミ_CSV/saketime_分類テスト用_analyzed.csv

📊 --- [構造化データ 品質レポート (全73件)] ---
 🔹 specific_name   : 取得済  32件 / 不明  41件 (欠損率 56.2%)
 🔹 rice_cultivar   : 取得済  14件 / 不明  59件 (欠損率 80.8%)
 🔹 body_type       : 取得済  52件 / 不明  21件 (欠損率 28.8%)
 🔹 amakara         : 取得済  53件 / 不明  20件 (欠損率 27.4%)
 🔹 sake_type       : 取得済  10件 / 不明  63件 (欠損率 86.3%)

💡 分析（Doc2Vec等）への活用準備が完全に整いました。
